You can view the content of the `kaggle.json` file by running the following command:

In [ ]:
import os

# 1. Set credentials directly in the environment
os.environ['KAGGLE_USERNAME'] = "syedasaniyakouser"
os.environ['KAGGLE_KEY'] = "KGAT_e46a03c0f0b90b582c705ba5bdaae3f8"

# 2. Install the kaggle tool
!pip install -q kaggle

# 3. Use the 'KaggleHub' method - it's a newer way that is often more stable
import kagglehub

# This downloads the dataset directly to the Colab cloud memory
print("Downloading dataset to Cloud...")
path = kagglehub.dataset_download("shaunthesheep/microsoft-catsvsdogs-dataset")

print(" Data downloaded to:", path)

# 4. List the folders to make sure we see 'PetImages'
os.listdir(path)

Using Colab cache for faster access to the 'microsoft-catsvsdogs-dataset' dataset.
 Data downloaded to: /kaggle/input/microsoft-catsvsdogs-dataset


['PetImages', 'readme[1].txt', 'MSR-LA - 3467.docx']

In [ ]:
import os
import cv2
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Setup the path from your previous output
data_dir = '/kaggle/input/microsoft-catsvsdogs-dataset/PetImages'
categories = ['Cat', 'Dog']
X, y = [], []

print(" Processing images from the Cloud... please wait.")

# 2. Load and Prepare Images
for category in categories:
    folder_path = os.path.join(data_dir, category)
    label = categories.index(category) # Cat = 0, Dog = 1

    # We take 400 images from each to keep it fast but accurate
    count = 0
    for img_name in os.listdir(folder_path):
        if count >= 400: break
        try:
            img_path = os.path.join(folder_path, img_name)
            # Read in Grayscale (faster math) and resize to small 50x50
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, (50, 50))

            X.append(img.flatten())
            y.append(label)
            count += 1
        except Exception as e:
            continue # Skip any corrupted images

X = np.array(X)
y = np.array(y)

# 3. Split the data for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Train the SVM Model
print(" Training the Support Vector Machine...")
model = SVC(kernel='linear')
model.fit(X_train, y_train)

# 5. Final Result
accuracy = accuracy_score(y_test, model.predict(X_test))
print(f"\n TASK 3 SUCCESS!")
print(f"Final Accuracy: {accuracy * 100:.2f}%")

 Processing images from the Cloud... please wait.
 Training the Support Vector Machine...

 TASK 3 SUCCESS!
Final Accuracy: 56.88%


In [24]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np

# 1. Create the GUI Elements
button = widgets.Button(description="Predict Random Image", button_style='info')
output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()
        # Pick a random image from your test set
        index = np.random.randint(0, len(X_test))
        test_img = X_test[index]
        true_label = "DOG" if y_test[index] == 1 else "CAT"

        # AI makes the prediction
        prediction = model.predict([test_img])
        pred_label = "DOG" if prediction[0] == 1 else "CAT"

        # AUTO-DETECT SIZE: This prevents the Reshape Error!
        side = int(len(test_img)**0.5)

        # Display the result
        plt.figure(figsize=(4,4))
        plt.imshow(test_img.reshape(side, side), cmap='gray')
        plt.title(f"True: {true_label} | AI Says: {pred_label}")
        plt.axis('off')
        plt.show()

        if true_label == pred_label:
            print(" The AI was correct!")
        else:
            print("The AI missed this one.")

button.on_click(on_button_clicked)

# 2. Display the GUI
print("--- Task 3: SVM Image Classifier GUI (Fixed) ---")
display(button, output)

--- Task 3: SVM Image Classifier GUI (Fixed) ---


Button(button_style='info', description='Predict Random Image', style=ButtonStyle())

Output()